# 02 — Unduh NSRDB Himawari 2016–2020 (T06)

Titik: **Kampus ITB Ganesha** (dari `configs/config.yaml`). Interval 10 menit, UTC, `leap_day=true`.

| Bagian | Isi | Checklist |
|---|---|---|
| 0–2 | Setup, kredensial, konfigurasi | T01.1 |
| 3 | Fungsi bantu (request, parse, validasi) | T06.1 |
| 4 | **Unduh 2016 sebagai uji** + isi elevasi ke config | T01.3, T06.2–T06.3 |
| 5 | Unduh 2017–2020 | T06.4 |
| 6 | Validasi semua tahun + MANIFEST md5 | T06.5 |

File yang sudah ada **tidak ditimpa** — aman dijalankan ulang.

## 0. Import & path

In [1]:
%pip install -q pandas pyyaml python-dotenv requests
import hashlib, io, json, os, re, time, getpass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, quote

import pandas as pd
import requests
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = ROOT / "configs" / "config.yaml"
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load(open(CONFIG_PATH))
print("ROOT =", ROOT)


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
ROOT = c:\Users\Asus\Downloads\ta-solar-forecast\ta-solar-forecast


## 1. Kredensial NLR (dari `.env`, atau diminta lewat input tersembunyi)

In [2]:
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
API_KEY = os.environ.get("NLR_API_KEY", "").strip() or getpass.getpass("NLR API key: ").strip()
EMAIL   = os.environ.get("NLR_EMAIL", "").strip()   or input("Email terdaftar NLR: ").strip()
assert API_KEY and EMAIL
print("Kredensial siap:", API_KEY[:4] + "…", EMAIL)

Kredensial siap: NJ7R… kharriskhisunica07@gmail.com


## 2. Konfigurasi request
Kalau API menolak salah satu atribut, pesan errornya muncul di bagian 4 — hapus atribut itu dari daftar lalu jalankan ulang (catat daftar final di `decisions.md`, T06.2).

In [7]:
ENDPOINT = "https://developer.nlr.gov/api/nsrdb/v2/solar/himawari-download.csv"
LAT, LON = CFG["location"]["lat"], CFG["location"]["lon"]
YEARS    = CFG["data"]["years"]
INTERVAL = int(CFG["data"]["nsrdb_interval_min"])

ATTRIBUTES = [
    "ghi", "dni", "dhi",
    "clearsky_ghi", "clearsky_dni", "clearsky_dhi",
    "solar_zenith_angle", "cloud_type",
    "air_temperature", "relative_humidity", "dew_point",
    "aod", "fill_flag",
]
print(f"Titik ({LAT}, {LON}) | {YEARS} | interval {INTERVAL} menit | {len(ATTRIBUTES)} atribut")

Titik (-6.8899, 107.61) | [2016, 2017, 2018, 2019, 2020] | interval 10 menit | 13 atribut


## 3. Fungsi bantu

In [4]:
def expected_rows(year, interval=INTERVAL):
    days = 366 if pd.Timestamp(year=year, month=12, day=31).dayofyear == 366 else 365
    return days * 24 * 60 // interval

def build_url(year, attributes=None):
    q = {
        "api_key": API_KEY, "wkt": f"POINT({LON} {LAT})", "names": year,
        "interval": INTERVAL, "utc": "true", "leap_day": "true",
        "email": EMAIL, "attributes": ",".join(attributes or ATTRIBUTES),
    }
    return ENDPOINT + "?" + urlencode(q, quote_via=quote)

def fetch_year(year, retries=3, wait=5):
    """GET satu tahun. Error 4xx dicetak (biasanya atribut/interval tidak valid) lalu dihentikan."""
    url = build_url(year)
    for k in range(1, retries + 1):
        r = requests.get(url, timeout=600)
        if r.status_code == 200 and not r.text.lstrip().startswith("{"):
            return r.text
        if r.status_code in (429, 500, 502, 503, 504):
            print(f"  {year}: HTTP {r.status_code}, coba lagi {k}/{retries} dalam {wait*k} dtk")
            time.sleep(wait * k); continue
        try:
            msg = json.dumps(r.json().get("errors", r.json()), indent=2)
        except ValueError:
            msg = r.text[:800]
        raise RuntimeError(f"{year}: HTTP {r.status_code}\n{msg}")
    raise RuntimeError(f"{year}: gagal setelah {retries} percobaan")

def parse_csv(text):
    """CSV NSRDB: baris 1 = nama metadata, baris 2 = nilai metadata, baris 3 = header data."""
    lines = text.splitlines()
    meta = dict(zip(lines[0].split(","), lines[1].split(",")))
    df = pd.read_csv(io.StringIO("\n".join(lines[2:])))
    df["time_utc"] = pd.to_datetime(df[["Year", "Month", "Day", "Hour", "Minute"]]).dt.tz_localize("UTC")
    return meta, df

def check_year(year, df):
    """Jumlah baris, duplikat, dan kontinuitas timestamp."""
    exp = expected_rows(year)
    dt = df["time_utc"].diff().dropna()
    res = {
        "tahun": year, "baris": len(df), "harapan": exp,
        "duplikat": int(df["time_utc"].duplicated().sum()),
        "lompatan": int((dt != pd.Timedelta(minutes=INTERVAL)).sum()),
        "awal": df["time_utc"].iloc[0], "akhir": df["time_utc"].iloc[-1],
    }
    res["OK"] = res["baris"] == exp and res["duplikat"] == 0 and res["lompatan"] == 0
    return res

def save_year(year, text):
    out = RAW / f"nsrdb_{year}.csv"
    if out.exists():
        print(f"  {out.name} sudah ada — tidak ditimpa"); return out
    out.write_text(text)
    return out

def md5(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

print("Fungsi siap.")

Fungsi siap.


## 4. Uji: unduh 2016 (T01.3, T06.3)

In [8]:
y0 = YEARS[0]
f0 = RAW / f"nsrdb_{y0}.csv"
text = f0.read_text() if f0.exists() else fetch_year(y0)
save_year(y0, text)
META, df0 = parse_csv(text)

print("HTTP 200 dan berisi data ✓")
print({k: META.get(k) for k in ("Source", "Location ID", "Latitude", "Longitude", "Elevation", "Time Zone", "Version")})
print("Kolom:", [c for c in df0.columns if c not in ("Year", "Month", "Day", "Hour", "Minute", "time_utc")])
print(f"{len(df0.columns) - 6} kolom data untuk {len(ATTRIBUTES)} atribut diminta (nama kolom NSRDB berbeda, mis. air_temperature → 'Temperature')")
print(check_year(y0, df0))
df0.head()

HTTP 200 dan berisi data ✓
{'Source': 'NSRDB', 'Location ID': '3106737', 'Latitude': '-6.88', 'Longitude': '107.61', 'Elevation': '770', 'Time Zone': '0', 'Version': 'unknown'}
Kolom: ['GHI', 'DNI', 'DHI', 'Clearsky GHI', 'Clearsky DNI', 'Clearsky DHI', 'Solar Zenith Angle', 'Cloud Type', 'Temperature', 'Relative Humidity', 'Dew Point', 'Aerosol Optical Depth', 'Fill Flag']
13 kolom data untuk 13 atribut diminta (nama kolom NSRDB berbeda, mis. air_temperature → 'Temperature')
{'tahun': 2016, 'baris': 52704, 'harapan': 52704, 'duplikat': 0, 'lompatan': 0, 'awal': Timestamp('2016-01-01 00:00:00+0000', tz='UTC'), 'akhir': Timestamp('2016-12-31 23:50:00+0000', tz='UTC'), 'OK': True}


,Year,Month,Day,Hour,Minute,GHI,DNI,DHI,Clearsky GHI,Clearsky DNI,Clearsky DHI,Solar Zenith Angle,Cloud Type,Temperature,Relative Humidity,Dew Point,Aerosol Optical Depth,Fill Flag,time_utc
0,2016,1,1,0,0,246,470,99,246,470,99,71.80,0,23.8,90.25,22.1,0.2366,0,2016-01-01 00:00:00+00:00
1,2016,1,1,0,10,285,509,107,285,509,107,69.51,0,23.9,89.71,22.1,0.2366,0,2016-01-01 00:10:00+00:00
2,2016,1,1,0,20,195,79,164,325,545,114,67.21,3,24.1,88.64,22.1,0.2366,0,2016-01-01 00:20:00+00:00
3,2016,1,1,0,30,365,577,121,365,577,121,64.92,1,24.2,88.12,22.1,0.2366,0,2016-01-01 00:30:00+00:00
4,2016,1,1,0,40,404,601,128,404,601,128,62.63,0,24.3,86.90,22.0,0.2424,0,2016-01-01 00:40:00+00:00


In [9]:
# Isi elevasi (header NSRDB) ke configs/config.yaml tanpa menghapus komentar
elev = float(META["Elevation"])
txt = CONFIG_PATH.read_text()
txt = re.sub(r"elevation_m:\s*\S+", f"elevation_m: {elev:g}", txt, count=1)
CONFIG_PATH.write_text(txt)
print(f"elevation_m = {elev:g} m → config.yaml")
print(f"Grid NSRDB terdekat: ({META['Latitude']}, {META['Longitude']}) vs titik ({LAT}, {LON})")

elevation_m = 770 m → config.yaml
Grid NSRDB terdekat: (-6.88, 107.61) vs titik (-6.8899, 107.61)


## 5. Unduh 2017–2020 (T06.4)

In [10]:
for y in YEARS[1:]:
    if (RAW / f"nsrdb_{y}.csv").exists():
        print(f"{y}: sudah ada"); continue
    print(f"{y}: mengunduh…")
    save_year(y, fetch_year(y))
    time.sleep(3)          # jeda antar-request
print("Selesai.")

2017: mengunduh…
2018: mengunduh…
2019: mengunduh…
2020: mengunduh…
Selesai.


## 6. Validasi semua tahun + MANIFEST (T06.5)

In [11]:
rows, manifest = [], [f"# NSRDB Himawari | POINT({LON} {LAT}) | interval={INTERVAL} | utc=true | leap_day=true | "
                      f"attributes={','.join(ATTRIBUTES)} | dibuat {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}"]
for y in YEARS:
    p = RAW / f"nsrdb_{y}.csv"
    if not p.exists():
        rows.append({"tahun": y, "OK": False, "catatan": "belum diunduh"}); continue
    _, d = parse_csv(p.read_text())
    rows.append(check_year(y, d))
    manifest.append(f"{md5(p)}  {p.name}  {p.stat().st_size}")
(RAW / "MANIFEST_nsrdb").write_text("\n".join(manifest) + "\n")
summary = pd.DataFrame(rows)
print("MANIFEST_nsrdb ditulis.")
print("SEMUA OK ✓" if summary["OK"].all() else "⚠ Ada tahun yang belum OK — lihat tabel")
summary

MANIFEST_nsrdb ditulis.
SEMUA OK ✓


,tahun,baris,harapan,duplikat,lompatan,awal,akhir,OK
0,2016,52704,52704,0,0,2016-01-01 00:00:00+00:00,2016-12-31 23:50:00+00:00,True
1,2017,52560,52560,0,0,2017-01-01 00:00:00+00:00,2017-12-31 23:50:00+00:00,True
2,2018,52560,52560,0,0,2018-01-01 00:00:00+00:00,2018-12-31 23:50:00+00:00,True
3,2019,52560,52560,0,0,2019-01-01 00:00:00+00:00,2019-12-31 23:50:00+00:00,True
4,2020,52704,52704,0,0,2020-01-01 00:00:00+00:00,2020-12-31 23:50:00+00:00,True


**Catat di `decisions.md`:** daftar atribut final, interval, grid NSRDB terdekat, elevasi. Lalu tandai T06 Selesai. Berikutnya: T08 (QC & agregasi per jam).